# Bronze — Cost Entries

Lê **todos** os CSVs de `landing/cost_entries/` e escreve Delta Lake.

**Estratégia:** `overwrite` completo (layout flat, sem particionamento Hive) — idempotente.  
**Nota:** os arquivos são nomeados por `usage_date` (não `execution_date`), cobrindo todos os meses históricos.  
**Colunas adicionadas:** `year` e `month` derivados de `usage_date` (colunas comuns, não de partição).

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15

In [ ]:
import sys
import os

year, month, day = int(year), int(month), int(day)

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if os.path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DecimalType, TimestampType, DateType
from pyspark.sql.utils import AnalysisException

In [ ]:
spark = create_spark_session("bronze_cost_entries")

# Lê todos os CSVs de landing — cada arquivo corresponde a um usage_date
# O nome do arquivo é cost_entries_YYYY-MM-DD.csv (não o execution_date)
input_path = "s3a://datalake/landing/cost_entries/cost_entries_*.csv"

try:
    df = spark.read.csv(input_path, header=True, inferSchema=False)
except AnalysisException:
    print(f"[bronze_cost_entries] Nenhum arquivo encontrado em landing/cost_entries/. Encerrando.")
    spark.stop()
    df = None

if df is not None:
    df_bronze = (
        df.select(
            F.col("id").cast(IntegerType()).alias("id"),
            F.col("team_id").cast(IntegerType()).alias("team_id"),
            F.col("resource_name").cast(StringType()).alias("resource_name"),
            F.col("resource_type").cast(StringType()).alias("resource_type"),
            F.col("provider").cast(StringType()).alias("provider"),
            F.col("region").cast(StringType()).alias("region"),
            F.col("environment").cast(StringType()).alias("environment"),
            F.col("usage_date").cast(DateType()).alias("usage_date"),
            F.col("cost_usd").cast(DecimalType(10, 4)).alias("cost_usd"),
            F.col("currency").cast(StringType()).alias("currency"),
            F.col("tags").cast(StringType()).alias("tags"),
            F.col("created_at").cast(TimestampType()).alias("created_at"),
        )
        .withColumn("year", F.year("usage_date"))
        .withColumn("month", F.month("usage_date"))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_layer", F.lit("landing"))
    )

    output_path = "s3a://datalake/bronze/cost_entries/"
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(output_path)
    )

    print(f"[bronze_cost_entries] {df_bronze.count()} linhas → {output_path}")
    df_bronze.groupBy("provider").count().orderBy("provider").show()
    spark.stop()